In [ ]:
%run ./utils_common

In [ ]:
logger = setup_logger("DBUCostReporter")

In [ ]:
dbutils.widgets.text("catalog", "", "CATALOG")
dbutils.widgets.text("schema", "", "SCHEMA")
dbutils.widgets.text("overlap_days", "3", "Overlap days (min 2)")
dbutils.widgets.text("workspace_ids", "", "Workspace IDs (comma-separated, blank=all)")
dbutils.widgets.text("price_unpriced_fail_pct", "", "Unpriced DBU fail threshold % (blank=1.0, negative=warn-only)")

In [ ]:
# =======================================================
# DBU Cost Client
# =======================================================
class DBUCostClient:

    TABLE_NAME = "dbspend360_dbu_cost"

    def __init__(self, audit_table: str, target_table: str, covered_table: str, overlap_days: int,
                 logger=None, price_fail_threshold_pct=_PRICE_UNPRICED_FAIL_PCT_DEFAULT):
        self.audit_table = audit_table
        self.target_table = target_table
        self.covered_table = covered_table
        self.overlap_days = overlap_days
        self.logger = logger or logging.getLogger("DBUCostClient")
        # Unpriced-DBU guard threshold (percent); None = warn-only. See
        # validate_price_coverage / get_price_fail_threshold in utils_common.
        self.price_fail_threshold_pct = price_fail_threshold_pct
        raw_ws = dbutils.widgets.get("workspace_ids")
        if raw_ws.strip() == "":
            self.workspace_ids = None
        else:
            self.workspace_ids = [w.strip() for w in raw_ws.split(",") if w.strip()]

    def compute_and_merge_dbu_cost(self):
        start_dt = end_dt = datetime.now(timezone.utc).date()
        try:
            start_dt, end_dt = get_date_window(self.audit_table, self.TABLE_NAME, self.overlap_days)

            valid, msg = validate_date_window(start_dt, end_dt)
            if not valid:
                raise DataQualityError(msg)

            self.logger.info(f"Loading DBU cost from {start_dt} to {end_dt}")

            cluster_df = (
                spark.table("system.compute.clusters")
                     .select("cluster_id", "cluster_name", "cluster_source", "workspace_id")
                     .filter("cluster_source = 'JOB'")
            )
            if self.workspace_ids is not None:
                cluster_df = cluster_df.filter(F.col("workspace_id").isin(self.workspace_ids))

            usage_df = (
                spark.table("system.billing.usage")
                     .alias("usage")
                     .filter(
                         (F.col("usage.usage_date") >= F.lit(start_dt)) &
                         (F.col("usage.usage_date") <= F.lit(end_dt))
                     )
            )
            if self.workspace_ids is not None:
                usage_df = usage_df.filter(F.col("usage.workspace_id").isin(self.workspace_ids))

            list_prices_df = spark.table("system.billing.list_prices").alias("list_prices")

            df = (
                usage_df.join(
                    list_prices_df,
                    on=(
                        (F.col("usage.sku_name") == F.col("list_prices.sku_name")) &
                        (F.col("usage.usage_start_time") >= F.col("list_prices.price_start_time")) &
                        (
                            (F.col("usage.usage_start_time") < F.col("list_prices.price_end_time")) |
                            F.col("list_prices.price_end_time").isNull()
                        )
                    ),
                    how="left"
                )
            )

            filtered_df = df.filter(
                F.col("usage.usage_metadata")["job_run_id"].isNotNull()
            )

            # Silent-undercount guard (mirrors the pipeline collector's
            # PRICE_JOIN_DROP guard): the LEFT join keeps usage rows
            # whose SKU/price-window has no list price, and SUM(qty * NULL price)
            # would drop that DBU with no error. Measured on filtered_df -- the
            # exact frame that feeds the aggregation -- so the reported unpriced
            # slice is the DBU that would be dropped from the SUM.
            _, coverage_summary = validate_price_coverage(
                filtered_df,
                price_col=F.col("list_prices.pricing")["default"],
                quantity_col=F.col("usage.usage_quantity"),
                sku_col=F.col("usage.sku_name"),
                table_context=self.target_table,
                logger=self.logger,
                fail_threshold_pct=self.price_fail_threshold_pct,
            )

            agg_df = (
                filtered_df
                .groupBy(
                    F.col("usage.usage_metadata")["cluster_id"].alias("job_cluster_id"),
                    F.col("usage.usage_metadata")["job_id"].alias("job_id"),
                    F.col("usage.usage_metadata")["job_run_id"].alias("run_id"),
                    F.col("usage.usage_date").alias("usage_date"),
                    F.col("usage.workspace_id").alias("workspace_id")
                )
                .agg(
                    F.sum(
                        F.col("usage.usage_quantity")
                        * F.col("list_prices.pricing")["default"].cast("double")
                    ).alias("databricks_cost"),
                    F.concat_ws(
                        " + ",
                        F.array_sort(F.collect_set(F.col("usage.sku_name")))
                    ).alias("sku_name_merged")
                )
            )

            job_cluster_df = (
                cluster_df
                .select("cluster_id")
                .dropDuplicates(["cluster_id"])
            )

            joined_df = (
                agg_df.join(
                    job_cluster_df,
                    on=(agg_df["job_cluster_id"] == job_cluster_df["cluster_id"]),
                    how="inner"
                )
                .drop("job_cluster_id")
            )

            joined_df = joined_df.withColumn("currency", F.lit("USD"))

            joined_df = add_workspace_covered(joined_df, self.covered_table, "workspace_id")

            dbu_inc_df = (
                joined_df
                .select(
                    "cluster_id",
                    "job_id",
                    "run_id",
                    "usage_date",
                    "databricks_cost",
                    "currency",
                    F.col("sku_name_merged").alias("sku_name"),
                    "workspace_id",
                    "workspace_covered",
                )
            )

            if dbu_inc_df.limit(1).count() == 0:
                self.logger.info("No DBU rows after filtering / aggregation.")
                merged_row_count = 0
            else:
                dbu_inc_df = (
                    dbu_inc_df
                    .withColumn("created_at", F.current_timestamp())
                    .withColumn("updated_at", F.current_timestamp())
                )
                dbu_inc_df = safe_cache(dbu_inc_df)

                merged_row_count = dbu_inc_df.count()

                validate_source_schema(
                    dbu_inc_df,
                    {"cluster_id": "string", "job_id": "string", "run_id": "string",
                     "usage_date": "date", "databricks_cost": "double"},
                    self.target_table, self.logger,
                )
                validate_no_negative_costs(
                    dbu_inc_df, ["databricks_cost"], self.target_table, self.logger,
                )
                validate_currency_consistency(dbu_inc_df, "currency", self.target_table, self.logger)

                ensure_boolean_columns(self.target_table, ["workspace_covered"], logger=self.logger)
                target = DeltaTable.forName(spark, self.target_table)
                (target.alias("t")
                    .merge(
                        dbu_inc_df.alias("s"),
                        "t.cluster_id = s.cluster_id AND t.job_id = s.job_id "
                        "AND t.run_id = s.run_id AND t.usage_date = s.usage_date",
                    )
                    .whenMatchedUpdate(set={
                        "databricks_cost": "s.databricks_cost",
                        "workspace_covered": "s.workspace_covered",
                        "updated_at": "current_timestamp()",
                    })
                    .whenNotMatchedInsert(values={
                        "cluster_id": "s.cluster_id",
                        "job_id": "s.job_id",
                        "run_id": "s.run_id",
                        "usage_date": "s.usage_date",
                        "databricks_cost": "s.databricks_cost",
                        "currency": "s.currency",
                        "sku_name": "s.sku_name",
                        "workspace_id": "s.workspace_id",
                        "workspace_covered": "s.workspace_covered",
                        "created_at": "current_timestamp()",
                        "updated_at": "current_timestamp()",
                    })
                    .execute()
                )

                safe_unpersist(dbu_inc_df)
                get_merge_metrics(self.target_table, self.logger)

                validate_post_merge(
                    self.target_table, "usage_date",
                    start_dt, end_dt, merged_row_count, self.logger,
                )

            log_audit_run(self.audit_table, self.TABLE_NAME, start_dt, end_dt, "SUCCESS", merged_row_count, coverage_summary)
            self.logger.info(
                f"Merged {merged_row_count} rows into {self.target_table} "
                f"for {start_dt} → {end_dt}."
            )

        except Exception as e:
            msg = str(e)[:1000]
            self.logger.error(f"Run failed: {msg}")
            try:
                log_audit_run(
                    self.audit_table, self.TABLE_NAME, start_dt, end_dt, "FAILED", 0, msg,
                )
            except Exception:
                self.logger.error("Failed to write FAILED audit entry")
            raise

In [ ]:
# =======================================================
# APP
# =======================================================
class DBUCostReporterApp:

    def __init__(self):
        catalog = dbutils.widgets.get("catalog")
        schema = dbutils.widgets.get("schema")
        overlap_days = get_overlap_days(dbutils.widgets.get("overlap_days"), logger=logger)
        price_fail_threshold_pct = get_price_fail_threshold(
            dbutils.widgets.get("price_unpriced_fail_pct"), logger=logger
        )

        audit_table = build_table_fqn(catalog, schema, "dbspend360_audit_log")
        target_table = build_table_fqn(catalog, schema, "dbspend360_dbu_cost")

        self.client = DBUCostClient(
            audit_table=audit_table,
            target_table=target_table,
            covered_table=build_table_fqn(catalog, schema, "dbspend360_covered_workspaces"),
            overlap_days=overlap_days,
            logger=logger,
            price_fail_threshold_pct=price_fail_threshold_pct,
        )

    def run(self):
        self.client.compute_and_merge_dbu_cost()

In [ ]:
# =======================================================
# Execute
# =======================================================
app = DBUCostReporterApp()
app.run()